In [ ]:
# 安装依赖
!pip install transformers torch pandas scikit-learn joblib

# 导入库
import pandas as pd #读取csv表格
import torch  #深度学习框架
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder

# 配置
MODEL_NAME = "hfl/chinese-bert-wwm-ext"
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5  #学习率
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 加载数据
train_df = pd.read_csv("train_data.csv")
val_df = pd.read_csv("val_data.csv")
test_df = pd.read_csv("test_data.csv")

# 标签编码
label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df["q_code"])
val_df["label"] = label_encoder.transform(val_df["q_code"])
test_df["label"] = label_encoder.transform(test_df["q_code"])

# 数据集类
class NOTAMDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, item):
        enc = self.tokenizer(str(self.texts[item]), max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"].flatten(), "attention_mask": enc["attention_mask"].flatten(), "label": torch.tensor(self.labels[item])}

# 初始化
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
train_loader = DataLoader(NOTAMDataset(train_df["e_text"], train_df["label"], tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(NOTAMDataset(val_df["e_text"], val_df["label"], tokenizer, MAX_LEN), batch_size=BATCH_SIZE)

# 模型
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(label_encoder.classes_)).to(device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader)*EPOCHS)

# 训练函数
def run(model, loader, opt, mode):
    model.train() if mode=="train" else model.eval()
    corr, loss = 0, 0 #loss损失值
    with torch.set_grad_enabled(mode=="train"):
        for b in loader:
            out = model(b["input_ids"].to(device), b["attention_mask"].to(device), labels=b["label"].to(device))
            corr += (torch.max(out.logits,1)[1] == b["label"].to(device)).sum()
            loss += out.loss.item()
            if mode=="train":
                out.loss.backward(); opt.step(); scheduler.step(); opt.zero_grad()
    return corr.double()/len(loader.dataset), loss/len(loader)

# 开始训练
best = 0
for e in range(EPOCHS):
    ta, tl = run(model, train_loader, optimizer, "train")
    va, vl = run(model, val_loader, optimizer, "val")
    print(f"轮{e+1} | 训练准确率:{ta:.2f} | 验证准确率:{va:.2f}")
    if va>best: best=va; torch.save(model.state_dict(), "best_model.bin")

# 测试
model.load_state_dict(torch.load("best_model.bin"))
test_loader = DataLoader(NOTAMDataset(test_df["e_text"], test_df["label"], tokenizer, MAX_LEN), batch_size=BATCH_SIZE)
test_acc, _ = run(model, test_loader, optimizer, "val")
print(f"\n最终测试准确率: {test_acc:.2f}")

In [ ]:
# ===================== 追加模块：计算并输出 召回率、F1、精确率 =====================
from sklearn.metrics import classification_report
import numpy as np

print("\n" + "="*80)
print(" 模型详细评估指标（精确率 / 召回率 / F1分数）")
print("="*80)

# 1. 收集测试集所有 真实标签 和 预测标签
y_true = []  # 真实Q码（数字）
y_pred = []  # 模型预测Q码（数字）

model.eval()  # 开启预测模式
with torch.no_grad():
    for batch in test_loader:
        # 数据加载
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # 模型预测
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=1)

        # 保存结果
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

# 2. 生成详细分类报告（直接显示Q码，而非数字）
# target_names = 所有Q码名称
report = classification_report(
    y_true,
    y_pred,
    target_names=label_encoder.classes_,
    digits=4  # 保留4位小数，更精准
)

# 3. 打印所有指标
print(report)

# 4. 简易文字解释
print("\n 指标解释：")
print("🔹 精确率(Precision)：预测为该Q码的样本中，真正正确的比例")
print("🔹 召回率(Recall)：真实为该Q码的样本中，被成功找出的比例（核心指标）")
print("🔹 F1-score：精确率+召回率的综合评分，越接近1越好")
print("🔹 support：该Q码在测试集中的实际样本数量")


 模型详细评估指标（精确率 / 召回率 / F1分数）
              precision    recall  f1-score   support

           A     0.9770    0.9551    0.9659        89
           F     0.9878    0.9000    0.9419        90
           I     0.9688    0.9725    0.9706       255
           K     1.0000    0.6667    0.8000         3
           L     1.0000    0.6667    0.8000         9
           M     0.9650    0.9972    0.9808       718
           N     0.9851    0.9296    0.9565        71
           O     1.0000    1.0000    1.0000        18
           P     1.0000    0.9000    0.9474        20
           R     0.9000    0.9000    0.9000        10
           S     1.0000    0.5000    0.6667         2
           W     1.0000    1.0000    1.0000        11
           X     0.8750    0.7500    0.8077        28

    accuracy                         0.9683      1324
   macro avg     0.9737    0.8567    0.9029      1324
weighted avg     0.9684    0.9683    0.9674      1324


 指标解释：
🔹 精确率(Precision)：预测为该Q码的样本中，真正正确的比例
🔹 召回率(

In [ ]:
# ===================== BERT 交互式预测（Colab 专用） =====================

def bert_predict():
    print("="*60)
    print("BERT 模型预测 - 输入 E 项文本，输出 Q 代码")
    print("="*60)

    # 1. 终端输入你的 E 项文本
    e_text = input("请输入/粘贴 E 项文本：")

    # 2. BERT 文本编码（和训练一致）
    inputs = tokenizer(
        e_text,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    # 3. 加载最优模型并预测
    model.load_state_dict(torch.load("best_model.bin", map_location=device))
    model.eval()

    with torch.no_grad():
        outputs = model(
            inputs["input_ids"].to(device),
            inputs["attention_mask"].to(device)
        )

    # 4. 转换为 Q 码
    pred_label = torch.argmax(outputs.logits, dim=1).cpu().item()
    q_code = label_encoder.inverse_transform([pred_label])[0]

    # 5. 输出结果
    print("\n 预测完成！")
    print(f"对应的 Q 代码为：【{q_code}】")
    print("="*60)

# 运行预测
bert_predict()

BERT 模型预测 - 输入 E 项文本，输出 Q 代码


KeyboardInterrupt: Interrupted by user